In [1]:
# CELL 1 — Imports & config

import requests
import os
import json
import time
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

NVD_API_KEY = os.getenv("NVD_API_KEY")
BASE_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
HEADERS = {"apiKey": NVD_API_KEY}
RESULTS_PER_PAGE = 2000  # NVD max allowed per request
DATA_DIR = "../data"

os.makedirs(DATA_DIR, exist_ok=True)
print(f"API Key loaded: {'✅' if NVD_API_KEY else '❌ NOT FOUND'}")

API Key loaded: ✅


In [2]:
# CELL 2 — Cleaning function

def clean_cve(raw):
    """
    Extract and validate fields from a raw NVD CVE object.
    Returns a clean dict or None if the CVE fails quality checks.
    """
    cve = raw["cve"]
    metrics = cve.get("metrics", {})

    # Must have CVSS v3.1
    if "cvssMetricV31" not in metrics:
        return None

    cvss = metrics["cvssMetricV31"][0]["cvssData"]

    # Must have a description in English
    descriptions = [d for d in cve.get("descriptions", []) if d["lang"] == "en"]
    if not descriptions or len(descriptions[0]["value"]) < 100:
        return None

    # Must have CPE (affected product info)
    configurations = cve.get("configurations", [])
    if not configurations:
        return None

    return {
        "id": cve["id"],
        "published": cve["published"],
        "lastModified": cve["lastModified"],
        "description": descriptions[0]["value"],
        "cvss_score": cvss["baseScore"],
        "cvss_severity": cvss["baseSeverity"],
        "attack_vector": cvss["attackVector"],
        "attack_complexity": cvss["attackComplexity"],
        "privileges_required": cvss["privilegesRequired"],
        "user_interaction": cvss["userInteraction"],
        "confidentiality_impact": cvss["confidentialityImpact"],
        "integrity_impact": cvss["integrityImpact"],
        "availability_impact": cvss["availabilityImpact"],
        "configurations": configurations,
        "references": [r["url"] for r in cve.get("references", [])]
    }

In [4]:
# CELL 3 — Quick API test (verify connectivity before running full pipeline)

params = {
    "resultsPerPage": 1,
    "startIndex": 0,
    "cvssV3Severity": "CRITICAL"
}
response = requests.get(BASE_URL, headers=HEADERS, params=params)

if response.status_code == 200:
    data = response.json()
    print(f"✅ API working")
    print(f"   Total CRITICAL CVEs available: {data['totalResults']}")
    
    time.sleep(0.6)  # NVD rate limit: 50 req/30s with API key
    
    print(f"   Total HIGH CVEs available: ", end="")
    # Quick check for HIGH too
    params_high = params.copy()
    params_high["cvssV3Severity"] = "HIGH"
    response_high = requests.get(BASE_URL, headers=HEADERS, params=params_high)
    data_high = response_high.json()
    print(f"{data_high['totalResults']}")
else:
    print(f"❌ API error {response.status_code}")


❌ API error 503


In [4]:
# CELL 3 — Pipeline config & helpers

GOALS = {"CRITICAL": 5000, "HIGH": 5000}
FILES = {
    "CRITICAL": f"{DATA_DIR}/cves_critical.jsonl",
    "HIGH": f"{DATA_DIR}/cves_high.jsonl",
}
PROGRESS_FILE = f"{DATA_DIR}/progress.json"


def load_progress():
    """Load the last NVD startIndex reached per severity."""
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {s: {"last_start": 0} for s in GOALS}

def save_progress(progress):
    with open(PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)

def load_existing(severity):
    """Return (count, set_of_ids) for already-stored clean CVEs."""
    path = FILES[severity]
    if not os.path.exists(path):
        return 0, set()
    seen_ids = set()
    count = 0
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                seen_ids.add(json.loads(line)["id"])
                count += 1
    return count, seen_ids


# Quick status check
for sev in GOALS:
    n, _ = load_existing(sev)
    print(f"{sev}: {n}/{GOALS[sev]} clean CVEs stored")

CRITICAL: 0/5000 clean CVEs stored
HIGH: 0/5000 clean CVEs stored


In [5]:
# CELL 4 — Batch fetch-clean-append pipeline
# Safe to re-run: resumes from the last saved startIndex, deduplicates by CVE ID

def run_pipeline(severity):
    goal = GOALS[severity]
    progress = load_progress()
    count, seen_ids = load_existing(severity)
    start = progress[severity]["last_start"]

    print(f"\n{'='*55}")
    print(f"[{severity}] Goal: {goal} | Stored: {count} | Resuming from startIndex {start}")

    if count >= goal:
        print(f"✅ Already at goal for {severity}")
        return count

    with open(FILES[severity], "a") as out:
        with tqdm(total=goal, initial=count, desc=severity, unit="CVE") as pbar:
            while count < goal:
                params = {
                    "resultsPerPage": RESULTS_PER_PAGE,
                    "startIndex": start,
                    "cvssV3Severity": severity,
                }
                r = requests.get(BASE_URL, headers=HEADERS, params=params)

                if r.status_code == 429:
                    print(f"\n⚠️  Rate limited — sleeping 30s")
                    time.sleep(30)
                    continue

                if r.status_code != 200:
                    print(f"\n❌ API error {r.status_code} at startIndex {start} — stopping")
                    break

                try:
                    data = r.json()
                except Exception:
                    print(f"\n⚠️  Empty/invalid response at startIndex {start} — sleeping 30s and retrying")
                    time.sleep(30)
                    continue

                total_available = data["totalResults"]
                batch = data.get("vulnerabilities", [])

                if not batch:
                    print(f"\n⚠️  Empty batch at startIndex {start} — exhausted source")
                    break

                for raw in batch:
                    cve_id = raw["cve"]["id"]
                    if cve_id in seen_ids:
                        continue
                    cleaned = clean_cve(raw)
                    if cleaned:
                        out.write(json.dumps(cleaned) + "\n")
                        out.flush()
                        seen_ids.add(cve_id)
                        count += 1
                        pbar.update(1)
                        if count >= goal:
                            break

                start += len(batch)
                progress[severity]["last_start"] = start
                save_progress(progress)

                if start >= total_available and count < goal:
                    print(f"\n⚠️  Exhausted all {total_available} available {severity} CVEs")
                    print(f"   Collected {count}/{goal} — source may not have enough passing the quality filter")
                    break

                time.sleep(0.6)  # NVD rate limit: 50 req/30s with API key

    final_count, _ = load_existing(severity)
    print(f"[{severity}] Done: {final_count} clean CVEs → {FILES[severity]}")
    return final_count

In [6]:
# CELL 5 — Run the pipeline
# Re-running this cell is safe: it picks up where it left off

for severity in ["CRITICAL", "HIGH"]:
    run_pipeline(severity)

print("\n--- Final tally ---")
for severity in ["CRITICAL", "HIGH"]:
    count, _ = load_existing(severity)
    goal = GOALS[severity]
    bar = "█" * int(count / goal * 20) + "░" * (20 - int(count / goal * 20))
    status = "✅" if count >= goal else f"{count}/{goal}"
    print(f"  {severity:8s} [{bar}] {status}")


[CRITICAL] Goal: 5000 | Stored: 0 | Resuming from startIndex 0


CRITICAL:   0%|                                                       | 0/5000 [00:00<?, ?CVE/s]



❌ API error 503 at startIndex 0 — stopping
[CRITICAL] Done: 0 clean CVEs → ../data/cves_critical.jsonl

[HIGH] Goal: 5000 | Stored: 0 | Resuming from startIndex 0


HIGH:   0%|                                                           | 0/5000 [00:00<?, ?CVE/s]


❌ API error 503 at startIndex 0 — stopping
[HIGH] Done: 0 clean CVEs → ../data/cves_high.jsonl

--- Final tally ---
  CRITICAL [░░░░░░░░░░░░░░░░░░░░] 0/5000
  HIGH     [░░░░░░░░░░░░░░░░░░░░] 0/5000
